In [ ]:

# --------------------------------------------
# Silver Layer Processing - Patient Dimension-
# --------------------------------------------
# Purpose:
# - Read streaming data from Bronze layer (Delta table)
# - Apply basic data quality rules (deduplication)
# - Mask sensitive information (PII protection using hashing)
# - Add audit column (load timestamp)
# - Perform incremental upsert (MERGE) into Silver Delta table
#
# Key Design:
# - Uses Structured Streaming with foreachBatch for micro-batch processing
# - Implements idempotent upsert logic using Delta MERGE
# - Supports schema evolution and scalable ingestion
# - Checkpointing ensures fault tolerance and exactly-once processing
# ------------------------------------------------------------------------
from delta.tables import DeltaTable
from pyspark.sql.window import Window 
from pyspark.sql.functions import (
    row_number, 
    sha2, 
    col, 
    current_timestamp, 
    monotonically_increasing_id, 
    concat_ws
)

silver_table = "DE_Learning_LH.silver.dim_patient"


df_patient_bronze = (
    spark.readStream.table("DE_Learning_LH.patient_bronze.patient")
)

df_patient_clean = (
    df_patient_bronze.select(
        "patient_id",
        "first_name",
        "last_name", 
        "gender", 
        col("city").alias("patient_city"), 
    ).withColumn(
        "masked_fullname",
        sha2(concat_ws('|', col("first_name"), col("last_name")), 256)
    ).withColumn("process_ts", current_timestamp()) 
)


def merge_dim_patient(batch_df, batch_id):
    if not spark.catalog.tableExists(silver_table):
        batch_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
        return

    # Load Delta table by name and upsert
    dim_patient = DeltaTable.forName(spark, silver_table)

    (
        dim_patient.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.patient_id = s.patient_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

(
    df_patient_clean.writeStream
        .foreachBatch(merge_dim_patient)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .option("checkpointLocation", "Files/healthcare_data/checkpoint/silver_patient/")
        .start()
)